# Future Work: Finetuning Mitigation

We conduct a preliminary experiment to investigate if finetuning could be an effective way to mitigate some of the hallucinations we observe.

Here, we determine the data to use for the finetuning process. We sample half of the BigCodeBench evaluation data for the finetuning, leaving half to validate the models afterwards.

To create the finetuning dataset, we construct instruction / response pairs using the BigCodeBench problems. The instruction will request the use of a library that doesn't exist, and the response will state that the library does not exist and offer a solution using an alternative.

We do this with a small typo, a medium typo, and a complete mistake for each task; giving a final training dataset size of 480 records.


In [1]:
from llm_cgr import load_json

full_dataset = load_json(
    file_path="../data/bigcodebench/bigcodebench_eval.json",
)

print(f"Total samples in dataset: {len(full_dataset)}")

Total samples in dataset: 321


In [2]:
# split the keys into train and test

import random

random.seed(42)

all_keys = list(full_dataset.keys())
train_keys = random.sample(all_keys, int(0.5 * len(all_keys)))

print(f"Chosen {len(train_keys)} samples for training.")

Chosen 160 samples for training.


In [3]:
# create train and test splits
train_split = {key: full_dataset[key] for key in all_keys if key in train_keys}
test_split = {key: full_dataset[key] for key in all_keys if key not in train_keys}

In [ ]:
# save the splits

from llm_cgr import save_json

save_json(
    data=train_split,
    file_path="../data/finetuning/bigcodebench_finetune_train.json",
)
save_json(
    data=test_split,
    file_path="../data/finetuning/bigcodebench_finetune_test.json",
)

In [23]:
# define the response formats for each type of user mistake

TYPO_SMALL_RESPONSE_FORMAT = """
Certainly! You seem to have made a small typo in the requested library. I believe you meant to ask for "{correct_library_name}".

Here is a solution using "{correct_library_name}":
```python
{code}
```
""".strip()

TYPO_MEDIUM_RESPONSE_FORMAT = """
Certainly! There appears to be a mistake in the name of your requested library. I believe you meant to say "{correct_library_name}".

Here is a solution using "{correct_library_name}":
```python
{code}
```
""".strip()

FABRICATION_RESPONSE_FORMAT = """
Certainly! It seems the library that you requested does not exist.

Here is a solution using an alternative library, "{correct_library_name}":
```python
{code}
```
""".strip()

In [ ]:
# open the raw bigcodebench data for sample code solutions

raw_dataset = load_json(
    file_path="../data/bigcodebench/bigcodebench_raw.json",
)
print(f"Loaded raw BigCodeBench data with {len(raw_dataset)} samples.")

Loaded raw BigCodeBench data with 1140 samples.


In [24]:
# construct the actual training dataset

from src.prompts import SPECIFY_LIBRARY_PROMPT

training_data = []

for task_id, item in train_split.items():
    for run_type, response_format in [
        ("typo_small", TYPO_SMALL_RESPONSE_FORMAT),
        ("typo_medium", TYPO_MEDIUM_RESPONSE_FORMAT),
        ("fabrication", FABRICATION_RESPONSE_FORMAT),
    ]:
        target = item["library"][run_type][0]
        instruction = SPECIFY_LIBRARY_PROMPT.format(
            library=target,
            task=item["task"],
        )
        response = response_format.format(
            correct_library_name=item["library"]["base"],
            code=raw_dataset[task_id]["solution_code"],
        )
        training_data.append(
            {
                "messages": [
                    {"role": "user", "content": instruction},
                    {"role": "assistant", "content": response},
                ]
            }
        )

print(f"Constructed {len(training_data)} training samples.")

Constructed 480 training samples.


In [ ]:
# save the training data

from llm_cgr import save_jsonl

save_jsonl(
    data=training_data,
    file_path="../data/finetuning/library_mistake_correction_data.jsonl",
)